# Validación de modelos y reporte de desempeño

Este notebook genera un **reporte HTML interactivo** (`reports/validacion_modelos.html`) para revisar el torneo de modelos con más detalle:

1. **Cómo se entrenó cada modelo**: todas las configuraciones de hiperparámetros probadas y el error en cada uno de los 10 folds de la validación cruzada.
2. **Comparación de métricas** en el conjunto de validación: RMSE real, NRMSE y RMSE logarítmico en train, CV y validación.
3. **Forma de la salida en test**: cada modelo se re-entrena con el 100 % de `train.csv` y predice los datos crudos de test. Sus predicciones se comparan con las del pipeline de inferencia del modelo ganador y con los precios reales de entrenamiento.
4. **Revisión de coherencia**: los modelos cuyas predicciones en test no son realistas se marcan como descartados.

El entrenamiento repite exactamente el del notebook `3.0-mls-torneo-modelos`: los mismos datos, la misma división, las mismas grillas y las mismas semillas.

## 1. Librerías

- **numpy**, **pandas**, **yaml** y **json**: manejo de datos, configuración y exportación de resultados.
- **joblib**: carga del pipeline de preprocesamiento y del pipeline de inferencia.
- **scipy.stats.gaussian_kde**: estimación de las curvas de densidad que se dibujan en el reporte.
- **scikit-learn**: modelos, validación cruzada, búsqueda de hiperparámetros y métricas.

In [1]:
# Librerías
import json
import os
import time
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from scipy.stats import gaussian_kde
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

## 2. Carga de datos, configuración y pipelines

Se cargan:

- Los **datos crudos** de entrenamiento y de test.
- La lista de features (`MODEL_FEATURES`).
- Una **copia sin ajustar** del pipeline de preprocesamiento, para re-ajustarlo dentro de cada fold y evitar fugas de datos, igual que en el torneo.
- El **pipeline de inferencia** guardado, que se usa como referencia para comparar la salida de cada modelo en test.

También se repite la división 80 % / 20 % con la misma semilla, para que el conjunto de validación sea idéntico al del torneo.

In [2]:
# Datos crudos, configuración, pipelines y división 80/20
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = Path(os.getenv("DATA_DIR", PROJECT_ROOT / "data" / "raw"))
MODELS_DIR = PROJECT_ROOT / "models"
CONFIG_FILE = PROJECT_ROOT / "pipeline_config.yaml"
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
TARGET = "SalePrice"

MODEL_FEATURES = yaml.safe_load(CONFIG_FILE.read_text(encoding="utf-8"))["MODEL_FEATURES"]
train = pd.read_csv(DATA_DIR / TRAIN_FILE)
test_raw = pd.read_csv(DATA_DIR / TEST_FILE)

preprocessing = clone(joblib.load(MODELS_DIR / "preprocessing_pipeline.joblib"))
inference_pipeline = joblib.load(MODELS_DIR / "inference_pipeline.joblib")

TEST_SIZE = 0.2
RANDOM_STATE = 42
X_train, X_val, y_train, y_val = train_test_split(
    train[MODEL_FEATURES], train[TARGET], test_size=TEST_SIZE, random_state=RANDOM_STATE
)
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

print(f"Entrenamiento: {X_train.shape} | Validación: {X_val.shape} | Test crudo: {test_raw.shape}")

Entrenamiento: (1168, 30) | Validación: (292, 30) | Test crudo: (1459, 80)


## 3. Modelos y grillas de hiperparámetros

Son los mismos seis modelos y las mismas grillas del notebook `3.0-mls-torneo-modelos`. **Si se cambian allá, deben cambiarse también aquí**, para que el reporte describa el mismo torneo. A cada modelo se le asigna su familia (regresión, ensamble o red neuronal) para mostrarla en el reporte.

In [3]:
# Modelos, familias y grillas (idénticos al notebook 3.0)
models = {
    "Ridge": (Ridge(), {"model__alpha": [0.1, 1, 10, 30, 100]}),
    "Lasso": (Lasso(max_iter=50_000), {"model__alpha": [0.0001, 0.0005, 0.001, 0.005, 0.01]}),
    "ElasticNet": (
        ElasticNet(max_iter=50_000),
        {"model__alpha": [0.0005, 0.001, 0.01], "model__l1_ratio": [0.2, 0.5, 0.8]},
    ),
    "RandomForest": (
        RandomForestRegressor(n_estimators=300, max_features=0.33, random_state=RANDOM_STATE, n_jobs=-1),
        {"model__max_depth": [8, 12, None], "model__min_samples_leaf": [1, 5, 10]},
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(learning_rate=0.03, subsample=0.8, random_state=RANDOM_STATE),
        {"model__n_estimators": [300, 600], "model__max_depth": [2, 3], "model__min_samples_leaf": [1, 5, 10]},
    ),
    "MLP": (
        TransformedTargetRegressor(
            regressor=MLPRegressor(max_iter=2000, early_stopping=True, n_iter_no_change=20, random_state=RANDOM_STATE),
            transformer=StandardScaler(),
        ),
        {"model__regressor__hidden_layer_sizes": [(64,), (64, 32), (128, 64)], "model__regressor__alpha": [0.001, 0.01, 0.1]},
    ),
}
FAMILIES = {
    "Ridge": "Regresión", "Lasso": "Regresión", "ElasticNet": "Regresión",
    "RandomForest": "Ensamble", "GradientBoosting": "Ensamble", "MLP": "Red neuronal",
}

## 4. Búsqueda de hiperparámetros con validación cruzada de 10 folds

Se repite la búsqueda del torneo: para cada modelo, `GridSearchCV` prueba todas sus configuraciones con 10 folds. El preprocesamiento se re-ajusta dentro de cada fold. Además del mejor resultado, se conservan los resultados completos (`cv_results_`): el error medio y su desviación para cada configuración, y el error de cada fold. Esos datos son los que el reporte usa para mostrar cómo se entrenó cada modelo.

Esta celda tarda unos minutos.

In [4]:
# GridSearchCV por modelo (misma configuración del torneo)
N_FOLDS = 10
cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

searches = {}
for name, (model, grid) in models.items():
    start = time.time()
    search = GridSearchCV(
        Pipeline(steps=[("preprocessing", clone(preprocessing)), ("model", model)]),
        param_grid=grid,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=-1,
    )
    search.fit(X_train, y_train_log)
    searches[name] = search
    print(f"{name:<17} RMSE CV (log): {-search.best_score_:.4f} | {time.time() - start:5.1f} s")

Ridge             RMSE CV (log): 0.1263 |   9.7 s


Lasso             RMSE CV (log): 0.1261 |   2.8 s


ElasticNet        RMSE CV (log): 0.1261 |   4.5 s


RandomForest      RMSE CV (log): 0.1326 |  25.3 s


GradientBoosting  RMSE CV (log): 0.1221 |  40.9 s


MLP               RMSE CV (log): 0.1316 |  15.5 s


## 5. Métricas en el conjunto de validación

Para cada modelo optimizado, ajustado con el 80 % de entrenamiento, se calcula:

- **RMSE real**: error en dólares sobre validación, después de regresar las predicciones con `expm1`.
- **NRMSE**: RMSE real dividido entre el rango de precios de entrenamiento.
- **RMSE log** en train, CV y validación, y la **brecha train → CV**, para evaluar el sobreajuste.

El ganador es el modelo con menor RMSE real en validación, con el mismo criterio del torneo.

In [5]:
# Métricas de validación por modelo
price_range = y_train.max() - y_train.min()

rows, val_predictions = [], {}
for name, search in searches.items():
    best = search.best_index_
    pred_val_log = search.predict(X_val)
    pred_val = np.expm1(pred_val_log)
    val_predictions[name] = pred_val
    rmse_real = root_mean_squared_error(y_val, pred_val)
    rmse_train_log = -search.cv_results_["mean_train_score"][best]
    rmse_cv_log = -search.best_score_
    rows.append({
        "modelo": name,
        "rmse_real": rmse_real,
        "nrmse": rmse_real / price_range,
        "rmse_log_train": rmse_train_log,
        "rmse_log_cv": rmse_cv_log,
        "rmse_log_val": root_mean_squared_error(y_val_log, pred_val_log),
        "brecha_train_cv_%": 100 * (rmse_cv_log - rmse_train_log) / rmse_train_log,
    })

results = pd.DataFrame(rows).sort_values("rmse_real").reset_index(drop=True)
winner_name = results.loc[0, "modelo"]
print(f"Ganador en validación: {winner_name}")
results.round(4)

Ganador en validación: GradientBoosting


,modelo,rmse_real,nrmse,rmse_log_train,rmse_log_cv,rmse_log_val,brecha_train_cv_%
0,GradientBoosting,27092.8257,0.0382,0.0715,0.1221,0.1359,70.7213
1,RandomForest,28753.0577,0.0405,0.0504,0.1326,0.1448,162.8405
2,Ridge,29853.5522,0.0420,0.1244,0.1263,0.1346,1.5545
3,MLP,29956.2540,0.0422,0.1006,0.1316,0.1385,30.8621
4,ElasticNet,30207.2879,0.0425,0.1242,0.1261,0.1344,1.5192
5,Lasso,30237.4821,0.0426,0.1243,0.1261,0.1344,1.4787


## 6. Re-entrenamiento con el 100 % y predicción sobre test crudo

Para comparar la salida de cada modelo con la del pipeline de inferencia, todos se preparan **igual que el ganador**:

1. Se toma una copia sin ajustar de su mejor configuración.
2. Se envuelve en `TransformedTargetRegressor`, que aplica `log1p` al entrenar y `expm1` al predecir.
3. Se re-entrena con todo `train.csv`.
4. Se predicen los datos crudos de test.

El pipeline de inferencia guardado también predice test. Como **control**, las predicciones del ganador re-entrenado aquí deben coincidir con las del pipeline de inferencia. Si no coinciden, el notebook de inferencia no está usando el mismo modelo.

In [6]:
# Re-entrenamiento de cada modelo con todo train.csv y predicción en test crudo
test_predictions = {}
for name, search in searches.items():
    final_model = TransformedTargetRegressor(
        regressor=clone(search.best_estimator_), func=np.log1p, inverse_func=np.expm1
    )
    final_model.fit(train[MODEL_FEATURES], train[TARGET])
    test_predictions[name] = final_model.predict(test_raw[MODEL_FEATURES])

inference_predictions = inference_pipeline.predict(test_raw)
inference_check_max_diff = np.abs(test_predictions[winner_name] - inference_predictions).max()
print(f"Control: diferencia máxima entre {winner_name} re-entrenado y el pipeline de inferencia: {inference_check_max_diff:,.4f}")

Control: diferencia máxima entre GradientBoosting re-entrenado y el pipeline de inferencia: 0.0000


## 7. Revisión de coherencia de la salida en test

Un modelo puede tener buenas métricas y aun así producir precios poco realistas con datos nuevos. Por eso, sus predicciones en test se someten a tres reglas. Si falla **cualquiera**, el modelo se **descarta**:

1. **Sin vacíos ni precios ≤ 0**: todas las viviendas deben tener un precio predicho positivo.
2. **Rango plausible**: todas las predicciones deben estar entre 0.5 × el precio mínimo y 1.5 × el precio máximo de entrenamiento. Un valor fuera de ese rango indica que el modelo extrapola de forma descontrolada.
3. **Mediana cercana**: la mediana predicha no debe alejarse más de ±15 % de la mediana de los precios de entrenamiento. Una desviación mayor indica que el modelo sobrestima o subestima sistemáticamente.

In [7]:
# Reglas de coherencia sobre las predicciones de test
RANGE_LOW_FACTOR, RANGE_HIGH_FACTOR = 0.5, 1.5
MEDIAN_TOLERANCE = 0.15

train_prices = train[TARGET]
min_allowed = RANGE_LOW_FACTOR * train_prices.min()
max_allowed = RANGE_HIGH_FACTOR * train_prices.max()
train_median = train_prices.median()
median_low, median_high = train_median * (1 - MEDIAN_TOLERANCE), train_median * (1 + MEDIAN_TOLERANCE)

coherence = {}
for name, pred in test_predictions.items():
    no_invalid = bool(np.isfinite(pred).all() and (pred > 0).all())
    in_range = bool(no_invalid and pred.min() >= min_allowed and pred.max() <= max_allowed)
    median_ok = bool(median_low <= np.median(pred) <= median_high)
    coherence[name] = {"no_invalid": no_invalid, "in_range": in_range, "median_ok": median_ok,
                       "coherent": no_invalid and in_range and median_ok}

coherence_table = pd.DataFrame(coherence).T.assign(
    mediana_predicha=[np.median(test_predictions[n]) for n in coherence],
    min_predicho=[np.min(test_predictions[n]) for n in coherence],
    max_predicho=[np.max(test_predictions[n]) for n in coherence],
)
print(f"Rango permitido: {min_allowed:,.0f} – {max_allowed:,.0f} | Mediana permitida: {median_low:,.0f} – {median_high:,.0f}")
coherence_table

Rango permitido: 17,450 – 1,132,500 | Mediana permitida: 138,550 – 187,450


,no_invalid,in_range,median_ok,coherent,mediana_predicha,min_predicho,max_predicho
Ridge,True,True,True,True,159485.639782,51389.225533,494676.423649
Lasso,True,True,True,True,159615.258539,50964.551047,496646.993090
ElasticNet,True,True,True,True,159709.306841,50887.657835,497002.431193
RandomForest,True,True,True,True,155801.929014,55080.381186,513951.582261
GradientBoosting,True,True,True,True,156263.708535,40533.520453,555089.298432
MLP,True,True,True,True,158546.972599,42068.362931,566522.336508


## 8. Preparación de los datos del reporte

Se reúnen en un único diccionario todos los datos que dibuja el reporte:

- Las métricas y la configuración elegida de cada modelo.
- Los resultados de todas las configuraciones de la búsqueda y el error de cada fold.
- Los pares precio real / predicho en validación.
- Las curvas de densidad de las predicciones en test.

Las curvas se calculan con una estimación de densidad por kernel (`gaussian_kde`) sobre una grilla de precios común, para que todas se puedan comparar en el mismo eje: cada modelo en test, el pipeline de inferencia en test y los precios reales de train. Los datos se guardan también en `reports/validacion_modelos_data.json`, por si se necesitan fuera del reporte.

In [8]:
# Construcción de los datos del reporte
GRID_POINTS = 200
grid_max = 1.05 * max(train_prices.max(), inference_predictions.max())
density_grid = np.linspace(0, grid_max, GRID_POINTS)
density = lambda values: gaussian_kde(values)(density_grid)

report_models = []
for _, row in results.iterrows():
    name = row["modelo"]
    cvr = searches[name].cv_results_
    best = searches[name].best_index_
    pred_test = test_predictions[name]
    report_models.append({
        "name": name,
        "family": FAMILIES[name],
        "best_params": ", ".join(f"{k}={v}" for k, v in searches[name].best_params_.items()),
        "rmse_real": row["rmse_real"],
        "nrmse": row["nrmse"],
        "rmse_log_train": row["rmse_log_train"],
        "rmse_log_cv": row["rmse_log_cv"],
        "rmse_log_val": row["rmse_log_val"],
        "gap_pct": row["brecha_train_cv_%"],
        "fold_scores": [-cvr[f"split{k}_test_score"][best] for k in range(N_FOLDS)],
        "configs": [
            {"params": ", ".join(f"{k}={v}" for k, v in params.items()),
             "mean": -cvr["mean_test_score"][i], "std": cvr["std_test_score"][i],
             "mean_train": -cvr["mean_train_score"][i], "rank": int(cvr["rank_test_score"][i])}
            for i, params in enumerate(cvr["params"])
        ],
        "val_scatter": [[round(float(r)), round(float(p))] for r, p in zip(y_val, val_predictions[name])],
        "test_density": density(pred_test[np.isfinite(pred_test)]).tolist(),
        "test_stats": {"min": float(np.min(pred_test)), "median": float(np.median(pred_test)), "max": float(np.max(pred_test))},
        "coherence": coherence[name],
    })

report_data = {
    "meta": {
        "generated": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "n_train": len(X_train), "n_val": len(X_val), "n_test": len(test_raw), "n_folds": N_FOLDS,
        "price_range": float(price_range), "train_median": float(train_median), "winner": winner_name,
        "inference_check_max_diff": float(inference_check_max_diff),
        "rules": {"min_allowed": float(min_allowed), "max_allowed": float(max_allowed),
                  "median_low": float(median_low), "median_high": float(median_high),
                  "median_tolerance_pct": int(MEDIAN_TOLERANCE * 100)},
    },
    "density_grid": density_grid.tolist(),
    "train_density": density(train_prices).tolist(),
    "inference_density": density(inference_predictions).tolist(),
    "models": report_models,
}

REPORTS_DIR = PROJECT_ROOT / "reports"
DATA_FILE = REPORTS_DIR / "validacion_modelos_data.json"
DATA_FILE.write_text(json.dumps(report_data, default=float), encoding="utf-8")
print(f"Datos del reporte guardados en: {DATA_FILE.relative_to(PROJECT_ROOT)}")

Datos del reporte guardados en: reports\validacion_modelos_data.json


## 9. Generación del reporte HTML

La plantilla del reporte (`house_pricing_a/templates/validacion_modelos.html`) contiene la estructura de la página y el código de las gráficas interactivas. Esta celda inserta los datos en la plantilla y escribe el resultado en `reports/validacion_modelos.html`.

El archivo es **autocontenido**: no necesita internet ni librerías externas. Se abre con doble clic en cualquier navegador. Las gráficas muestran el detalle de cada valor al pasar el mouse, y cada sección incluye una tabla con los datos exactos.

In [9]:
# Inserción de los datos en la plantilla y escritura del reporte
TEMPLATE_FILE = PROJECT_ROOT / "house_pricing_a" / "templates" / "validacion_modelos.html"
REPORT_FILE = REPORTS_DIR / "validacion_modelos.html"

template = TEMPLATE_FILE.read_text(encoding="utf-8")
report_html = template.replace("/*__DATA__*/null", json.dumps(report_data, default=float))
REPORT_FILE.write_text(report_html, encoding="utf-8")

descartados = [m["name"] for m in report_models if not m["coherence"]["coherent"]]
print(f"Reporte generado en: {REPORT_FILE.relative_to(PROJECT_ROOT)}")
print(f"Modelos descartados por coherencia: {descartados or 'ninguno'}")

Reporte generado en: reports\validacion_modelos.html
Modelos descartados por coherencia: ninguno
